# Kafka Silver — Transformation

## Transformation Decision Report

### Purpose

This notebook transforms the Kafka Bronze interaction events into the Silver layer.

The transformation decisions are based on the findings from the Kafka Bronze EDA.

The goal is to improve technical consistency, normalize event fields, expose data-quality issues explicitly, and preserve the original event-level grain and business information.

## Transformation Principles

- Preserve one Silver record for each source event.
- Do not delete events because they contain dirty or imperfect business values.
- Preserve original identifiers and relationships.
- Apply technical normalization where it is safe and deterministic.
- Convert fields to appropriate analytical data types.
- Preserve questionable values when they remain useful for analysis.
- Represent detected data-quality problems through explicit flags.
- Do not silently impute missing values.
- Do not change interaction meaning based on assumptions.
- Preserve event timestamps when they can be parsed successfully.
- Preserve invalid timestamp records and flag them rather than deleting them.

## Event Decisions

| Area | Decision |
|---|---|
| event_id | Preserve unchanged |
| event_timestamp | Parse from string into timestamp |
| invalid event_timestamp | Preserve original event and flag invalid timestamp |
| user_id | Preserve unchanged |
| video_id | Preserve unchanged |
| interaction_type | Normalize whitespace/case where technically safe |
| unknown interaction types | Preserve original value and flag |
| watch_time_sec | Preserve original value |
| missing watch_time_sec | Preserve as NULL |
| watch_time <= 0 | Preserve and flag |
| watch_time > 120 | Preserve and flag |
| interaction/watch-time mismatch | Preserve and flag |
| device_type | Normalize whitespace/case where technically safe |
| unexpected device type | Preserve and flag |

## Timestamp Decisions

`event_timestamp` is currently stored as a string in Kafka Bronze.

Silver will attempt to parse it into a proper timestamp.

Invalid timestamp values such as `not-a-timestamp` will not cause the event to be deleted.

The invalid record will remain in Silver with an explicit data-quality flag.

Derived temporal fields such as event date and event hour may be created from the successfully parsed timestamp.

## Data Quality Flags

The Silver dataset should expose the following conditions where applicable:

- missing or invalid event timestamp
- unexpected interaction type
- invalid watch time
- watch time above the expected threshold
- interaction/watch-time inconsistency
- unexpected device type
- missing user ID
- missing video ID

The flags describe the quality of the source data and do not remove the underlying event.

## Expected Output

The Kafka Silver dataset will preserve the original event-level grain.

It will contain normalized analytical fields, parsed timestamps, derived temporal fields where valid, and explicit data-quality flags.

The transformation must preserve the relationship between:

User
+
Video
↓
Interaction Event

No source event should be silently discarded because of business-data-quality issues.

In [1]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 17:38:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-30-Sep"



KAFKA_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

In [3]:
from pyspark.sql.types import *

print("=" * 70)
print("KAFKA BRONZE — EDA")
print("=" * 70)



df_kafka = spark.read.parquet(KAFKA_BRONZE_PATH)

print("Kafka Bronze loaded successfully.")

KAFKA BRONZE — EDA


26/09/30 17:38:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Kafka Bronze loaded successfully.


In [4]:
df_kafka.show(1)

+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+
|            event_id|     event_timestamp|             user_id|            video_id|interaction_type|watch_time_sec|device_type|
+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+
|01a076ee-6c3d-787...|2026-01-01 00:00:...|01a076ee-6c38-778...|01a076ee-4b63-784...|            like|          NULL|         tv|
+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+
only showing top 1 row



In [4]:


VALID_INTERACTIONS = (
    "like",
    "play",
    "complete",
    "pause",
)

VALID_DEVICES = (
    "mobile",
    "tv",
    "web",
)

df_kafka_silver = (
    df_kafka

    # --------------------------------------------------
    # Preserve source timestamp
    # --------------------------------------------------

    .withColumn(
        "event_timestamp_raw",
        F.col("event_timestamp")
    )

    # --------------------------------------------------
    # Technical normalization
    # --------------------------------------------------

    .withColumn(
        "interaction_type",
        F.lower(F.trim(F.col("interaction_type")))
    )

    .withColumn(
        "device_type",
        F.lower(F.trim(F.col("device_type")))
    )

    .withColumn(
        "event_timestamp",
        F.to_timestamp(F.col("event_timestamp"))
    )

    # --------------------------------------------------
    # Derived temporal fields
    # --------------------------------------------------

    .withColumn(
        "event_date",
        F.to_date(F.col("event_timestamp"))
    )

    .withColumn(
        "event_hour",
        F.hour(F.col("event_timestamp"))
    )

    # --------------------------------------------------
    # Data Quality Flags
    # --------------------------------------------------

    .withColumn(
        "dq_missing_user_id",
        F.col("user_id").isNull()
        | (F.trim(F.col("user_id")) == "")
    )

    .withColumn(
        "dq_missing_video_id",
        F.col("video_id").isNull()
        | (F.trim(F.col("video_id")) == "")
    )

    .withColumn(
        "dq_missing_watch_time",
        F.col("interaction_type").isin("play", "complete")
        & F.col("watch_time_sec").isNull()
    )

    .withColumn(
        "dq_unexpected_watch_time",
        F.col("interaction_type").isin("like", "pause")
        & F.col("watch_time_sec").isNotNull()
    )

    .withColumn(
        "dq_invalid_watch_time",
        F.col("watch_time_sec").isNotNull()
        & (
            (F.col("watch_time_sec") <= 0)
            | (F.col("watch_time_sec") > 120)
        )
    )

    .withColumn(
        "dq_invalid_interaction",
        F.col("interaction_type").isNull()
        | ~F.col("interaction_type").isin(*VALID_INTERACTIONS)
    )

    .withColumn(
        "dq_invalid_device",
        F.col("device_type").isNull()
        | ~F.col("device_type").isin(*VALID_DEVICES)
    )

    .withColumn(
        "dq_invalid_timestamp",
        F.col("event_timestamp").isNull()
    )

    # --------------------------------------------------
    # Metadata
    # --------------------------------------------------

    .withColumn(
        "ingestion_date",
        F.current_date()
    )

    .withColumn(
        "batch_id",
        F.concat(
            F.date_format(
                F.current_timestamp(),
                "yyyyMMdd_HHmmss"
            ),
            F.lit("_kafka")
        )
    )
)

In [5]:
df_kafka_silver.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")